In [11]:
import sys
sys.path.append('/kaggle/input/datasets/pankajdeopa/vr-project-config/')
import config

import os, torch, numpy as np, pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Paths to processed data
PROCESSED_DIR = '/kaggle/input/datasets/pankajdeopa/vr-processed-data/'
CSV_PATH      = PROCESSED_DIR + 'master_labels.csv'
PW_PATH       = PROCESSED_DIR + 'pos_weight.pt'

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 5
BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LR          = 1e-4

IDX_TO_NAME = {0: 'short_sleeve_top', 1: 'trousers', 2: 'shorts',
               3: 'long_sleeve_top',  4: 'skirt'}

os.makedirs('/kaggle/working/checkpoints/', exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [12]:
class ApparelDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df         = df.reset_index(drop=True)
        self.transform  = transform
        self.label_cols = ['label_0','label_1','label_2','label_3','label_4']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row[self.label_cols].values.astype(np.float32))
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

df       = pd.read_csv(CSV_PATH)
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']

train_loader = DataLoader(ApparelDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(ApparelDataset(val_df,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 144174 | Val: 23741


In [13]:
# build resnet model

def build_resnet50(pretrained=True):
    model = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_resnet50(pretrained=True).to(DEVICE)
print(f"ResNet-50 loaded | Output layer: {model.fc}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 206MB/s]


ResNet-50 loaded | Output layer: Linear(in_features=2048, out_features=5, bias=True)


In [17]:
# Training Setup

pos_weight = torch.load(PW_PATH).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [19]:
from tqdm import tqdm

def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Evaluating', leave=False):
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())

    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds      = (all_probs >= threshold).astype(int)

    macro_f1 = f1_score(all_labels, preds, average='macro',  zero_division=0)
    micro_f1 = f1_score(all_labels, preds, average='micro',  zero_division=0)
    macro_p  = precision_score(all_labels, preds, average='macro',  zero_division=0)
    macro_r  = recall_score(all_labels, preds, average='macro',  zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs, average='macro')
    except:
        auc = 0.0

    return macro_f1, micro_f1, macro_p, macro_r, auc


best_f1 = 0.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss  = 0.0
    train_bar   = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]")

    for imgs, labels in train_bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

        # Update progress bar with current loss
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_loader)
    scheduler.step()

    # ── Validate ──
    macro_f1, micro_f1, macro_p, macro_r, auc = evaluate(model, val_loader)

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'macro_f1': macro_f1, 'micro_f1': micro_f1,
        'precision': macro_p, 'recall': macro_r, 'auc': auc
    })

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
          f"Macro-F1: {macro_f1:.4f} | Micro-F1: {micro_f1:.4f} | AUC: {auc:.4f}")

    # ── Save best model ──
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), '/kaggle/working/checkpoints/resnet50_best.pt')
        print(f"  ✓ Best model saved (Macro-F1: {best_f1:.4f})")

print("\nTraining complete!")

Epoch 01/10 [Train]: 100%|██████████| 4506/4506 [28:37<00:00,  2.62it/s, loss=0.3497]


Epoch 01/10 | Loss: 0.4559 | Macro-F1: 0.8144 | Micro-F1: 0.8324 | AUC: 0.9573
  ✓ Best model saved (Macro-F1: 0.8144)


Epoch 02/10 [Train]: 100%|██████████| 4506/4506 [28:32<00:00,  2.63it/s, loss=0.3231]


Epoch 02/10 | Loss: 0.3666 | Macro-F1: 0.8299 | Micro-F1: 0.8473 | AUC: 0.9629
  ✓ Best model saved (Macro-F1: 0.8299)


Epoch 03/10 [Train]: 100%|██████████| 4506/4506 [28:30<00:00,  2.63it/s, loss=0.5221]


...

Epoch 08/10 | Loss: 0.1595 | Macro-F1: 0.8637 | Micro-F1: 0.8765 | AUC: 0.9707
  ✓ Best model saved (Macro-F1: 0.8637)


Epoch 09/10 [Train]: 100%|██████████| 4506/4506 [28:26<00:00,  2.64it/s, loss=0.1097]


Epoch 09/10 | Loss: 0.1399 | Macro-F1: 0.8670 | Micro-F1: 0.8780 | AUC: 0.9694
  ✓ Best model saved (Macro-F1: 0.8670)


Epoch 10/10 [Train]: 100%|██████████| 4506/4506 [28:26<00:00,  2.64it/s, loss=0.0660]


Epoch 10/10 | Loss: 0.1274 | Macro-F1: 0.8671 | Micro-F1: 0.8788 | AUC: 0.9700
  ✓ Best model saved (Macro-F1: 0.8671)

Training complete!


In [20]:
# Save history
pd.DataFrame(history).to_csv('/kaggle/working/resnet50_history.csv', index=False)

# Per-class evaluation on val set
model.load_state_dict(torch.load('/kaggle/working/checkpoints/resnet50_best.pt'))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)
preds      = (all_probs >= 0.5).astype(int)

print("Per-class Results (ResNet-50):")
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 55)
for i, name in IDX_TO_NAME.items():
    p   = precision_score(all_labels[:, i], preds[:, i], zero_division=0)
    r   = recall_score(all_labels[:, i], preds[:, i], zero_division=0)
    f   = f1_score(all_labels[:, i], preds[:, i], zero_division=0)
    auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
    print(f"{name:<20} {p:>10.4f} {r:>10.4f} {f:>10.4f} {auc:>10.4f}")

Per-class Results (ResNet-50):
Class                 Precision     Recall         F1        AUC
-------------------------------------------------------
short_sleeve_top         0.9011     0.9036     0.9023     0.9622
trousers                 0.8720     0.9480     0.9084     0.9806
shorts                   0.7707     0.8785     0.8211     0.9725
long_sleeve_top          0.7936     0.8453     0.8186     0.9575
skirt                    0.8793     0.8904     0.8849     0.9775
